# Detección de Fraude en Transacciones en Tiempo Real

## Objetivos de Aprendizaje

En este notebook, aprenderás a:

1. **Entrenar un clasificador binario** con `SNOWFLAKE.ML.CLASSIFICATION`
2. **Generar datos sintéticos** de transacciones bancarias con patrones de fraude
3. **Calcular z-scores y ratios** como features de detección de anomalías
4. **Aplicar umbrales de decisión** diferenciados (BLOQUEAR/REVISAR/APROBAR)
5. **Construir un dashboard interactivo** de monitorización con Streamlit

---

## Lo Que Construirás

Un **sistema de detección de fraude en tiempo real** que clasifica transacciones:
- Modelo de clasificación binaria (fraude/legítimo) sobre 50.000 transacciones
- Feature engineering con z-score de importe por cliente, dispositivo nuevo, ubicación anómala
- Umbrales de acción: BLOQUEAR (≥80%), REVISAR (≥50%), APROBAR (<50%)
- Dashboard de monitorización con KPIs y tabla de alertas

**Valor de Negocio**: Reducir pérdidas por fraude minimizando falsos positivos que bloquean clientes legítimos.

---

## Requisitos Técnicos

- **Cuenta Snowflake** con acceso SQL estándar
- **Cortex ML habilitado**
- **Aproximadamente 15 minutos** para completar
- **100% SQL** (excepto dashboard Streamlit)

---

## Funcionalidades Clave de Snowflake

- `SNOWFLAKE.ML.CLASSIFICATION` — Entrenamiento de modelo de clasificación
- `GENERATOR()` — Generación de datos sintéticos realistas
- `OBJECT_CONSTRUCT` — Predicción inline
- `STDDEV()` y `AVG()` — Cálculo de z-scores estadísticos
- `UNIFORM()` y `RANDOM()` — Distribuciones aleatorias para datos sintéticos
- Streamlit — Dashboard interactivo integrado

¡Comencemos!

---

## Paso 1: Configuración del Entorno

### Qué Vamos a Crear

- **Database**: `FRAUDE_TRANSACCIONES_DB` — Contenedor de todas las tablas y modelos
- **Schema**: `PUBLIC` — Namespace dentro de la base de datos
- **Warehouse**: `COMPUTE_WH` — Recursos de cómputo para consultas

### Por Qué Importa

Usar objetos dedicados mantiene este tutorial aislado del entorno de producción y facilita la limpieza.

### Conceptos de Snowflake

- **Database**: Contenedor lógico de esquemas (como una carpeta)
- **Schema**: Namespace para tablas, vistas y modelos ML
- **Warehouse**: Cluster de cómputo virtual que ejecuta consultas (puede auto-suspenderse cuando está inactivo)

In [ ]:
-- Crear base de datos dedicada para detección de fraude
CREATE DATABASE IF NOT EXISTS FRAUDE_TRANSACCIONES_DB;
CREATE SCHEMA IF NOT EXISTS FRAUDE_TRANSACCIONES_DB.PUBLIC;
USE SCHEMA FRAUDE_TRANSACCIONES_DB.PUBLIC;

-- Crear warehouse de cómputo
CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH 
    WITH WAREHOUSE_SIZE = 'SMALL'
    AUTO_SUSPEND = 60
    AUTO_RESUME = TRUE;

USE WAREHOUSE COMPUTE_WH;

-- Verificar configuración
SELECT 
    CURRENT_DATABASE() AS database_name,
    CURRENT_SCHEMA() AS schema_name,
    CURRENT_WAREHOUSE() AS warehouse_name;

---

## Paso 2: Generar Datos de Clientes

### Qué Vamos a Crear

**2.000 clientes bancarios** con perfiles demográficos realistas.

### La Función `GENERATOR()`

`GENERATOR(ROWCOUNT => N)` es la función de Snowflake para crear datos sintéticos:
- Genera N filas instantáneamente
- Combinada con `UNIFORM()`, `SEQ4()` y funciones de fecha para crear patrones realistas
- ¡No se necesitan archivos de datos externos!

### Distribución de Datos

- **Edad**: 22-75 años (distribución uniforme)
- **Ciudades**: 10 ciudades españolas principales
- **Ingresos**: 15.000€ — 120.000€ anuales
- **Antigüedad**: 1-20 años como cliente
- **Productos**: 1-5 productos contratados

In [ ]:
-- Generar 2.000 clientes bancarios
CREATE OR REPLACE TABLE CLIENTES (
    cliente_id VARCHAR(10) PRIMARY KEY,
    edad INTEGER,
    genero VARCHAR(1),
    ciudad VARCHAR(50),
    ingresos_anuales DECIMAL(10,2),
    antiguedad_anos INTEGER,
    num_productos INTEGER
);

INSERT INTO CLIENTES
SELECT
    'CLI' || LPAD(SEQ4()::STRING, 5, '0') AS cliente_id,
    UNIFORM(22, 75, RANDOM()) AS edad,
    CASE WHEN UNIFORM(0,1,RANDOM()) < 0.52 THEN 'M' ELSE 'F' END AS genero,
    ARRAY_CONSTRUCT('Madrid','Barcelona','Valencia','Sevilla','Bilbao',
                    'Zaragoza','Málaga','Murcia','Palma','Las Palmas')
        [UNIFORM(0,9,RANDOM())]::VARCHAR AS ciudad,
    ROUND(UNIFORM(15000, 120000, RANDOM()), 2) AS ingresos_anuales,
    UNIFORM(1, 20, RANDOM()) AS antiguedad_anos,
    UNIFORM(1, 5, RANDOM()) AS num_productos
FROM TABLE(GENERATOR(ROWCOUNT => 2000));

-- Vista previa de clientes generados
SELECT * FROM CLIENTES LIMIT 10;

---

## Paso 3: Generar Transacciones con Patrones de Fraude

### Qué Vamos a Crear

**50.000 transacciones** donde el 5% son fraudulentas, con señales distinguibles:

### Patrones de Fraude vs Legítimo

| Característica | Legítimo | Fraude |
|---|---|---|
| Importe | 5-800€ | 500-15.000€ |
| Ubicación | Ciudades españolas | Lagos, Bucarest, Kiev... |
| Hora local | 6:00-23:00 | 0:00-5:00 |
| Dispositivo nuevo | Raro | 70% de los casos |
| Canal | Predomina POS | Predomina Online |

### Funciones Clave

- **`CASE WHEN`**: Crear patrones diferenciados por clase
- **`ARRAY_CONSTRUCT`**: Pools de valores para selección aleatoria
- **CTE (`WITH`)**: Generar datos base y luego enriquecer

In [ ]:
-- Generar 50.000 transacciones (5% fraude)
CREATE OR REPLACE TABLE TRANSACCIONES (
    transaccion_id VARCHAR(12) PRIMARY KEY,
    cliente_id VARCHAR(10),
    fecha_hora TIMESTAMP,
    canal VARCHAR(20),
    importe DECIMAL(12,2),
    ubicacion VARCHAR(50),
    hora_local INTEGER,
    dispositivo_nuevo BOOLEAN,
    es_fraude BOOLEAN,
    FOREIGN KEY (cliente_id) REFERENCES CLIENTES(cliente_id)
);

INSERT INTO TRANSACCIONES
WITH base AS (
    SELECT
        'TXN' || LPAD(SEQ4()::STRING, 7, '0') AS transaccion_id,
        'CLI' || LPAD(UNIFORM(1, 2000, RANDOM())::STRING, 5, '0') AS cliente_id,
        DATEADD('minute', -UNIFORM(0, 525600, RANDOM()), CURRENT_TIMESTAMP()) AS fecha_hora,
        CASE UNIFORM(1,10,RANDOM())
            WHEN 1 THEN 'ATM' WHEN 2 THEN 'POS' WHEN 3 THEN 'Online'
            WHEN 4 THEN 'Transferencia' WHEN 5 THEN 'POS' WHEN 6 THEN 'Online'
            WHEN 7 THEN 'POS' WHEN 8 THEN 'Online' WHEN 9 THEN 'POS'
            ELSE 'Transferencia'
        END AS canal,
        CASE WHEN UNIFORM(0,100,RANDOM()) < 5 THEN TRUE ELSE FALSE END AS es_fraude
    FROM TABLE(GENERATOR(ROWCOUNT => 50000))
)
SELECT
    b.transaccion_id, b.cliente_id, b.fecha_hora, b.canal,
    CASE WHEN b.es_fraude 
         THEN ROUND(UNIFORM(500, 15000, RANDOM()) + UNIFORM(0,100,RANDOM())/100, 2)
         ELSE ROUND(UNIFORM(5, 800, RANDOM()) + UNIFORM(0,100,RANDOM())/100, 2) 
    END AS importe,
    CASE WHEN b.es_fraude 
         THEN ARRAY_CONSTRUCT('Lagos','Bucarest','Kiev','Manila','Estambul')[UNIFORM(0,4,RANDOM())]::VARCHAR
         ELSE ARRAY_CONSTRUCT('Madrid','Barcelona','Valencia','Sevilla','Bilbao','Online')[UNIFORM(0,5,RANDOM())]::VARCHAR 
    END AS ubicacion,
    CASE WHEN b.es_fraude THEN UNIFORM(0, 5, RANDOM()) 
         ELSE UNIFORM(6, 23, RANDOM()) END AS hora_local,
    CASE WHEN b.es_fraude AND UNIFORM(0,1,RANDOM()) < 0.7 THEN TRUE ELSE FALSE END AS dispositivo_nuevo,
    b.es_fraude
FROM base b;

-- Verificar distribución de fraude
SELECT 
    es_fraude,
    COUNT(*) AS n_transacciones,
    ROUND(AVG(importe), 2) AS importe_medio,
    ROUND(AVG(hora_local), 1) AS hora_media
FROM TRANSACCIONES 
GROUP BY 1;

---

## Paso 4: Ingeniería de Variables (Feature Engineering)

### Qué Vamos a Crear

Transformar datos brutos en **features predictivas** para el modelo ML:

1. **Z-Score de Importe**: Desviación del importe respecto al histórico del cliente
2. **Ratio vs Media**: Proporción del importe actual vs media histórica
3. **Indicadores de Canal**: Codificación one-hot (Online, ATM)
4. **Estadísticas del Cliente**: Canales y ubicaciones distintas usadas
5. **Demográficos**: Edad, antigüedad, productos del cliente

### Window Functions y Agregaciones

Usamos `AVG()` y `STDDEV()` agrupados por cliente para calcular **z-scores**:
```sql
(importe - AVG(importe)) / STDDEV(importe)  -- z-score por cliente
```

### Por Qué Importa el Feature Engineering

Los datos brutos (importes, fechas) no son útiles directamente para ML. Las features (z-scores, ratios, tendencias) son lo que los modelos aprenden.

In [ ]:
-- Crear features de detección de fraude
CREATE OR REPLACE TABLE FEATURES_FRAUDE AS
WITH stats_cliente AS (
    SELECT 
        cliente_id,
        AVG(importe) AS media_importe_historico,
        STDDEV(importe) AS std_importe_historico,
        COUNT(*) AS total_transacciones,
        COUNT(DISTINCT canal) AS canales_distintos,
        COUNT(DISTINCT ubicacion) AS ubicaciones_distintas
    FROM TRANSACCIONES 
    GROUP BY 1
)
SELECT
    t.transaccion_id,
    t.importe::FLOAT AS importe,
    t.hora_local::FLOAT AS hora_local,
    t.dispositivo_nuevo::FLOAT AS dispositivo_nuevo,
    CASE WHEN t.canal = 'Online' THEN 1 ELSE 0 END::FLOAT AS es_online,
    CASE WHEN t.canal = 'ATM' THEN 1 ELSE 0 END::FLOAT AS es_atm,
    (t.importe / NULLIF(s.media_importe_historico, 0))::FLOAT AS ratio_vs_media,
    CASE WHEN s.std_importe_historico > 0 
         THEN ((t.importe - s.media_importe_historico) / s.std_importe_historico)::FLOAT 
         ELSE 0 
    END AS zscore_importe,
    s.canales_distintos::FLOAT AS canales_distintos,
    s.ubicaciones_distintas::FLOAT AS ubicaciones_distintas,
    c.edad::FLOAT AS edad,
    c.antiguedad_anos::FLOAT AS antiguedad_anos,
    c.num_productos::FLOAT AS num_productos,
    t.es_fraude
FROM TRANSACCIONES t
JOIN stats_cliente s ON t.cliente_id = s.cliente_id
JOIN CLIENTES c ON t.cliente_id = c.cliente_id;

-- Vista previa de features
SELECT * FROM FEATURES_FRAUDE LIMIT 10;

---

## Paso 5: Preparar Datos de Entrenamiento

### Qué Vamos a Hacer

Dividir las 50.000 transacciones en:
- **Entrenamiento** (80%) — Para entrenar el modelo
- **Test** (20%) — Para evaluar el rendimiento

### ¿Por Qué Dividir los Datos?

Necesitamos evaluar el modelo con **datos no vistos** para asegurar que generaliza bien. Entrenar y evaluar con los mismos datos daría una precisión artificialmente alta.

### La Función `SAMPLE`

`SAMPLE (80)` selecciona aleatoriamente el 80% de las filas para entrenamiento. El 20% restante será nuestro conjunto de test.

In [ ]:
-- Dividir en entrenamiento (80%) y test (20%)
CREATE OR REPLACE TABLE ENTRENAMIENTO AS 
SELECT * FROM FEATURES_FRAUDE SAMPLE (80);

CREATE OR REPLACE TABLE TEST AS 
SELECT * FROM FEATURES_FRAUDE 
WHERE transaccion_id NOT IN (SELECT transaccion_id FROM ENTRENAMIENTO);

-- Verificar distribución
SELECT 
    'ENTRENAMIENTO' AS conjunto, 
    COUNT(*) AS n, 
    SUM(es_fraude::INT) AS fraudes,
    ROUND(SUM(es_fraude::INT) / COUNT(*) * 100, 1) || '%' AS tasa_fraude
FROM ENTRENAMIENTO
UNION ALL
SELECT 
    'TEST', COUNT(*), SUM(es_fraude::INT),
    ROUND(SUM(es_fraude::INT) / COUNT(*) * 100, 1) || '%'
FROM TEST;

---

## Paso 6: Entrenar el Modelo de Clasificación

### Qué Vamos a Crear

Un **modelo de clasificación binaria** que predice si una transacción es fraudulenta basándose en las features de comportamiento.

### Entendiendo `SNOWFLAKE.ML.CLASSIFICATION`

Función **AutoML nativa** de Snowflake que entrena modelos ML usando solo SQL:

```sql
CREATE SNOWFLAKE.ML.CLASSIFICATION model_name(
    INPUT_DATA => SYSTEM$REFERENCE('TABLE', 'ENTRENAMIENTO'),
    TARGET_COLNAME => 'ES_FRAUDE'
);
```

### ¿Por Qué ML.CLASSIFICATION para Fraude?

- **AutoML**: Prueba automáticamente XGBoost, Random Forest, Logistic Regression
- **Sin Python**: 100% SQL — accesible para analistas de negocio
- **Selección Automática de Features**: Identifica las variables más predictivas
- **Evaluación Integrada**: Prueba el rendimiento en datos de holdout
- **Escalable**: Maneja millones de transacciones sin cambios de código

### Cómo Funciona (Internamente)

1. **Analiza los datos**: Distribuciones, valores faltantes, correlaciones
2. **Entrena múltiples algoritmos**: 3-5 algoritmos en paralelo
3. **Ajuste de hiperparámetros**: Optimiza automáticamente
4. **Selecciona el mejor**: Elige el de mayor precisión
5. **Crea función de predicción**: Devuelve modelo con método `!PREDICT()`

In [ ]:
-- Entrenar modelo de clasificación de fraude
CREATE OR REPLACE SNOWFLAKE.ML.CLASSIFICATION detector_fraude_txn(
    INPUT_DATA => SYSTEM$REFERENCE('TABLE', 'ENTRENAMIENTO'),
    TARGET_COLNAME => 'ES_FRAUDE'
);

---

## Paso 7: Evaluar Rendimiento del Modelo

### Métricas Clave

- **Accuracy**: % de predicciones correctas (objetivo: >90%)
- **Precision**: De las transacciones marcadas como fraude, % que realmente lo son
- **Recall**: De las transacciones fraudulentas, % que detectamos
- **F1 Score**: Media armónica de precision y recall

### Matriz de Confusión

| | Predicho: Legítimo | Predicho: Fraude |
|---|---|---|
| **Real: Legítimo** | Verdadero Negativo (TN) | Falso Positivo (FP) |
| **Real: Fraude** | Falso Negativo (FN) | Verdadero Positivo (TP) |

### ¿Qué es "Suficientemente Bueno"?

Para detección de fraude bancario:
- **Alto recall** es crítico (no perder transacciones fraudulentas)
- **Precisión moderada** es aceptable (algunas falsas alarmas son OK)
- **Trade-off**: Reducir falsos negativos vs no bloquear clientes legítimos

In [ ]:
-- Métricas de evaluación del modelo
CALL detector_fraude_txn!SHOW_EVALUATION_METRICS();

-- Matriz de confusión
CALL detector_fraude_txn!SHOW_CONFUSION_MATRIX();

-- Importancia de variables
CALL detector_fraude_txn!SHOW_FEATURE_IMPORTANCE();

---

## Paso 8: Puntuar Transacciones y Aplicar Umbrales

### Qué Vamos a Hacer

Usar el modelo entrenado para **puntuar transacciones** del conjunto de test.

### El Método `!PREDICT()`

Los modelos ML de Snowflake exponen un método `!PREDICT()` que:
1. Toma features de una tabla
2. Devuelve predicciones en un objeto VARIANT:
   - **`class`**: Clase predicha (TRUE/FALSE)
   - **`probability`**: Confianza para cada clase

### Estratificación por Umbral

Clasificamos transacciones en niveles de acción:
- **BLOQUEAR**: probabilidad ≥80% → Acción inmediata
- **REVISAR**: probabilidad ≥50% → Revisión manual
- **APROBAR**: probabilidad <50% → Transacción normal

In [ ]:
-- Puntuar test y aplicar umbrales de decisión
CREATE OR REPLACE TABLE PREDICCIONES_FRAUDE AS
SELECT 
    transaccion_id,
    importe,
    hora_local,
    es_online,
    zscore_importe,
    es_fraude AS fraude_real,
    detector_fraude_txn!PREDICT(
        OBJECT_CONSTRUCT(
            'IMPORTE', importe, 'HORA_LOCAL', hora_local, 
            'DISPOSITIVO_NUEVO', dispositivo_nuevo,
            'ES_ONLINE', es_online, 'ES_ATM', es_atm, 
            'RATIO_VS_MEDIA', ratio_vs_media,
            'ZSCORE_IMPORTE', zscore_importe, 
            'CANALES_DISTINTOS', canales_distintos,
            'UBICACIONES_DISTINTAS', ubicaciones_distintas, 
            'EDAD', edad,
            'ANTIGUEDAD_ANOS', antiguedad_anos, 
            'NUM_PRODUCTOS', num_productos
        )
    ) AS prediccion,
    prediccion['class']::BOOLEAN AS fraude_predicho,
    ROUND(prediccion['probability']['true']::FLOAT * 100, 1) AS prob_fraude_pct,
    CASE
        WHEN prediccion['probability']['true']::FLOAT >= 0.8 THEN 'BLOQUEAR'
        WHEN prediccion['probability']['true']::FLOAT >= 0.5 THEN 'REVISAR'
        ELSE 'APROBAR'
    END AS accion
FROM TEST;

-- Resumen por acción
SELECT 
    accion,
    COUNT(*) AS n_transacciones,
    ROUND(AVG(prob_fraude_pct), 1) AS prob_media,
    SUM(CASE WHEN fraude_real THEN 1 ELSE 0 END) AS fraudes_reales
FROM PREDICCIONES_FRAUDE
GROUP BY 1 ORDER BY 2 DESC;

---

## Paso 9: Dashboard Interactivo

### Lo Que Verás

Un **dashboard Streamlit** integrado con:
- KPIs de monitorización (total, sospechosas, bloqueadas)
- Gráfico de distribución por acción
- Tabla de transacciones sospechosas con probabilidad

### Cómo Funciona

El dashboard está **embebido en el notebook** — no necesita despliegue adicional.

In [ ]:
import streamlit as st
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()

st.title('Detección de Fraude en Transacciones')
st.markdown('### Monitorización en Tiempo Real')

total = session.sql('SELECT COUNT(*) FROM PREDICCIONES_FRAUDE').collect()[0][0]
fraudes = session.sql("SELECT COUNT(*) FROM PREDICCIONES_FRAUDE WHERE prob_fraude_pct >= 50").collect()[0][0]
bloqueados = session.sql("SELECT COUNT(*) FROM PREDICCIONES_FRAUDE WHERE accion = 'BLOQUEAR'").collect()[0][0]

c1, c2, c3 = st.columns(3)
c1.metric('Total Transacciones', f'{total:,}')
c2.metric('Sospechosas', f'{fraudes:,}')
c3.metric('Bloqueadas', f'{bloqueados:,}')

st.markdown('---')
st.subheader('Distribución por Acción')
df_accion = session.sql("SELECT accion, COUNT(*) AS n FROM PREDICCIONES_FRAUDE GROUP BY 1 ORDER BY n DESC").to_pandas()
st.bar_chart(df_accion.set_index('ACCION'))

st.markdown('---')
st.subheader('Transacciones Sospechosas')
filtro = st.selectbox('Filtrar por acción', ['Todas','BLOQUEAR','REVISAR'])
if filtro == 'Todas':
    q = "SELECT transaccion_id, importe, prob_fraude_pct, accion FROM PREDICCIONES_FRAUDE WHERE prob_fraude_pct >= 50 ORDER BY prob_fraude_pct DESC LIMIT 50"
else:
    q = f"SELECT transaccion_id, importe, prob_fraude_pct, accion FROM PREDICCIONES_FRAUDE WHERE accion = '{filtro}' ORDER BY prob_fraude_pct DESC LIMIT 50"
st.dataframe(session.sql(q).to_pandas())

---

## Paso 10: Limpieza del Entorno (Opcional)

### Qué Hace Esto

Esta celda **eliminará completamente** todos los objetos creados:
- Elimina la base de datos `FRAUDE_TRANSACCIONES_DB` y todas sus tablas/modelos

### Instrucciones

**Esta celda está comentada por defecto** para prevenir eliminaciones accidentales.

Para limpiar:
1. **Descomenta** las líneas SQL
2. **Ejecuta** esta celda manualmente

⚠️ **Advertencia**: Esta acción no se puede deshacer.

In [ ]:
-- Descomentar las líneas siguientes para limpiar el entorno

-- DROP DATABASE IF EXISTS FRAUDE_TRANSACCIONES_DB;